# Cyanite starter: model outputs + search (with audio)

A runnable tour for the challenge:

1. **Model outputs (tagging)**: `GET /v1/library-tracks/{id}/models`
2. **Free-text search**: `POST /v1/private-alpha/library-tracks/search`
3. **Similarity search**: `POST /v1/private-alpha/library-tracks/{id}/similar`

Search returns track ids; fetch each result's tags via model outputs and explain the match.
Inline **audio players** (public Jamendo MP3s) are shown for results. See
[../guides/model_outputs.md](../guides/model_outputs.md) and
[../guides/tag_vocabularies.md](../guides/tag_vocabularies.md).

> Search is a **private alpha** feature (under `/private-alpha/`); shapes may change. Mind the
> shared rate limits and quotas (see the repo README).

**Setup:** `pip install requests python-dotenv`; put `CYANITE_API_KEY` in a `.env` (git-ignored).


In [ ]:
import os
import csv
import json
import tempfile
import requests
import IPython.display as ipd
from dotenv import load_dotenv

load_dotenv()  # reads CYANITE_API_KEY from .env
API_KEY = os.environ["CYANITE_API_KEY"]
BASE_URL = "https://rest-api.cyanite.ai/v1"

session = requests.Session()
session.headers.update({"x-api-key": API_KEY})
print("ready")


## Pick a track from the data pack

`data/tracks.csv` carries both ids: `track_id` (Jamendo) and `cyanite_id` (`libtr_...`, what
the API uses). We load a Jamendo -> Cyanite map and grab a sample to play with.


In [ ]:
def _tracks_path():
    for p in ("../data/tracks.csv", "data/tracks.csv"):
        if os.path.exists(p):
            return p
    raise FileNotFoundError("data/tracks.csv not found; run from the repo root or notebooks/")

JAMENDO_TO_CYANITE = {}
with open(_tracks_path()) as f:
    for row in csv.DictReader(f):
        JAMENDO_TO_CYANITE[row["track_id"]] = row["cyanite_id"]

def resolve_track_id(jamendo_id):
    # data-pack Jamendo id -> Cyanite track id
    return JAMENDO_TO_CYANITE[str(jamendo_id)]

SAMPLE_TRACK_ID = next(iter(JAMENDO_TO_CYANITE.values()))
print(len(JAMENDO_TO_CYANITE), "tracks loaded; sample cyanite id:", SAMPLE_TRACK_ID)


## Listen to a track

`play()` renders an inline audio player from the public Jamendo MP3 (downloaded and cached in
`/tmp`). It accepts a Jamendo id, a Cyanite id (`libtr_...`), or a **search-result item** (whose
`title` is the Jamendo filename, so any result is playable). It displays itself, so it works in
loops too.


In [ ]:
CYANITE_TO_JAMENDO = {c: j for j, c in JAMENDO_TO_CYANITE.items()}
AUDIO_DIR = os.path.join(tempfile.gettempdir(), "cyanite_audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

def jamendo_mp3_url(jamendo_id):
    return f"https://prod-1.storage.jamendo.com/download/track/{jamendo_id}/mp32/"

def to_jamendo_id(track):
    # accepts a Jamendo id, a Cyanite id, or a search-result item dict
    if isinstance(track, dict):
        if track.get("title"):
            return str(track["title"]).split(".mp3")[0]   # title is the Jamendo filename
        if track.get("externalId"):
            return str(track["externalId"])
        track = track.get("id")
    track = str(track)
    if track.startswith("libtr_"):
        return CYANITE_TO_JAMENDO.get(track)              # data-pack tracks only
    return track

def play(track, title=None):
    # render an inline audio player (downloads the public Jamendo MP3, cached)
    jam = to_jamendo_id(track)
    if not jam:
        print("No Jamendo audio for", track)
        return
    label = title or (track.get("title") if isinstance(track, dict) else None)
    if label:
        print(label)
    path = os.path.join(AUDIO_DIR, f"{jam}.mp3")
    if not os.path.exists(path):
        r = requests.get(jamendo_mp3_url(jam), timeout=60)
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    ipd.display(ipd.Audio(filename=path))

play(SAMPLE_TRACK_ID)


## 1. Model outputs (tagging)

Pass one or more `model` query params to choose which outputs to return. Full list in the model
outputs guide; the defaults below are the most useful for taste / explainability.


In [ ]:
DEFAULT_MODELS = [
    "MainGenreV2", "SubgenreV2", "FreeGenreV3",
    "MoodSimpleV2", "MoodAdvancedV2",
    "InstrumentsV2", "CharacterV2", "MovementV2",
    "BpmV2", "TempoV1", "KeyV2", "TimeSignatureV2",
    "ValenceArousalV2", "MusicalEraV2", "MusicForV1",
    "VocalsV2", "AutoDescriptionV2", "RepresentativeSegmentV2",
]

def get_model_outputs(track_id, models=None):
    # fetch selected model outputs for a Cyanite track id (libtr_...)
    models = models or DEFAULT_MODELS
    params = [("model", m) for m in models]
    resp = session.get(f"{BASE_URL}/library-tracks/{track_id}/models", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

outputs = get_model_outputs(SAMPLE_TRACK_ID, ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "AutoDescriptionV2"])
print(json.dumps(outputs, indent=2)[:2500])


### Summarize the tags

Pull the human-readable bits out for explanations. Adjust the accessors once you see a real
response (the model outputs guide documents each model's fields).


In [ ]:
def iter_model_outputs(outputs):
    if isinstance(outputs, dict):
        for key in ("data", "models", "results"):
            if isinstance(outputs.get(key), list):
                yield from outputs[key]
                return
        for v in outputs.values():
            if isinstance(v, dict) and "version" in v:
                yield v
        return
    if isinstance(outputs, list):
        yield from outputs

def summarize(outputs):
    # compact dominant tags / values per model, for "why this track" explanations
    summary = {}
    for mo in iter_model_outputs(outputs):
        version = mo.get("version", "?")
        if mo.get("tags") is not None:
            summary[version] = mo["tags"]
        elif "tag" in mo:
            summary[version] = mo["tag"]
        elif "description" in mo:
            summary[version] = mo["description"]
    return summary

summarize(outputs)


## 2. Free-text search  (private alpha)

Describe what you want in natural language; get back the best-matching tracks. Optional
`metadataFilter` narrows candidates before ranking. `limit` is 1-50 (default 20).

`show_results()` prints the ranked list and renders audio players for the top few.


In [ ]:
def free_text_search(query, limit=20, metadata_filter=None):
    # POST /private-alpha/library-tracks/search -> {items, pageInfo}
    body = {"query": query}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/search",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

def show_results(res, list_n=10, play_n=3):
    # print the ranked results, then render players for the top few
    items = res.get("items", [])
    for it in items[:list_n]:
        print(it.get("id"), "|", it.get("title"))
    print("nextCursor:", res.get("pageInfo", {}).get("nextCursor"))
    for it in items[:play_n]:
        play(it)


In [ ]:
res = free_text_search("dreamy ambient piano, slow and emotional", limit=10)
show_results(res)


In [ ]:
# narrow with a metadata filter (e.g. BPM range). operators: $gte $lte $eq $in
res = free_text_search(
    "energetic workout music",
    limit=10,
    metadata_filter={"BpmV2.tag": {"$gte": 120, "$lte": 140}},
)
show_results(res)


## 3. Similarity search  (private alpha)

Start from a track (a `libtr_...` id, e.g. a user's liked track resolved via the data pack) and
get the most similar tracks. Same `{items, pageInfo}` shape; `metadataFilter` optional.


In [ ]:
def similarity_search(track_id, limit=20, metadata_filter=None):
    # POST /private-alpha/library-tracks/{id}/similar -> {items, pageInfo}
    body = {}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/{track_id}/similar",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

res = similarity_search(SAMPLE_TRACK_ID, limit=10)
show_results(res)


## 4. The full loop: search -> tags -> explain (and listen)

Search for ids, fetch model outputs for a result, summarize its tags, and play it. This is the
core of an explainable recommendation.


In [ ]:
res = free_text_search("warm acoustic folk, gentle fingerpicked guitar", limit=5)
top = res["items"][0]
print("top result:", top["id"], "|", top.get("title"))
print(summarize(get_model_outputs(top["id"], ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "CharacterV2"])))
play(top)


## Where to go next

- Build a **taste profile** from a user's likes (`data/users.csv`): resolve their Jamendo ids to
  Cyanite ids, aggregate their tags, then drive free-text or similarity search.
- Re-rank similarity results for **tag coherence** and explain each pick via shared tags.
- Add **metadata filters** to steer along mood / genre / tempo axes.
- Cache model outputs locally and fetch each track once (shared rate limits / quotas).
